In [5]:
import pandas as pd
import numpy as np
from scipy import stats


# ============================================================
# 1. LOAD CLEANED DATA
# ============================================================

customers = pd.read_csv("cleaned_customers.csv")
orders = pd.read_csv("cleaned_orders.csv")
order_items = pd.read_csv("cleaned_order_items.csv")
reviews = pd.read_csv("cleaned_order_reviews.csv")
products = pd.read_csv("cleaned_products.csv")


# ============================================================
# 2. CREATE ORDER-LEVEL REVENUE
# ============================================================

order_items["item_revenue"] = (
    order_items["quantity"]
    * order_items["unit_price"]
    * (1 - order_items["discount(%)"] / 100)
)

order_revenue = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_revenue=("item_revenue", "sum")
    )
)


# ============================================================
# 3. CREATE ANALYSIS DATASET
# ============================================================

analysis = orders.merge(
    order_revenue,
    on="order_id",
    how="inner"
)

analysis = analysis.merge(
    customers[["customer_id", "customer_segment"]],
    on="customer_id",
    how="left"
)

print("Analysis dataset shape:", analysis.shape)


# ============================================================
# HELPER FUNCTION: NORMALITY CHECK
# ============================================================

def check_normality(data, group_name):

    data = data.dropna()

    # Shapiro-Wilk is not practical on very large samples.
    # Take a random sample of maximum 5000 observations.
    sample = data.sample(
        min(len(data), 5000),
        random_state=42
    )

    statistic, p_value = stats.shapiro(sample)

    print(f"\nNormality Check - {group_name}")
    print("Sample size used:", len(sample))
    print("Shapiro-Wilk statistic:", statistic)
    print("P-value:", p_value)

    if p_value >= 0.05:
        print("Result: Normality assumption is not rejected.")
        return True
    else:
        print("Result: Normality assumption is rejected.")
        return False


# ============================================================
# HELPER FUNCTION: SAMPLE SIZE CHECK
# ============================================================

def check_sample_size(data, group_name):

    n = data.dropna().shape[0]

    print(f"\nSample Size Check - {group_name}")
    print("N =", n)

    if n >= 30:
        print("Result: Sample size is adequate (n >= 30).")
        return True
    else:
        print("Result: Sample size is small (n < 30).")
        return False


# ============================================================
# H1
# NEW vs LOYAL CUSTOMER REVENUE
# Welch Independent t-test
# ============================================================

new = analysis.loc[
    analysis["customer_segment"].str.strip().str.lower() == "new",
    "order_revenue"
].dropna()

loyal = analysis.loc[
    analysis["customer_segment"].str.strip().str.lower() == "loyal",
    "order_revenue"
].dropna()

print("\n" + "=" * 70)
print("H1: NEW vs LOYAL CUSTOMER REVENUE")
print("=" * 70)

# Independence
print("\nIndependence assumption:")
print("Each order is treated as an independent observation.")

# Sample size
check_sample_size(new, "New Customers")
check_sample_size(loyal, "Loyal Customers")

# Normality
check_normality(new, "New Customers")
check_normality(loyal, "Loyal Customers")

# Welch t-test
t_h1, p_h1 = stats.ttest_ind(
    new,
    loyal,
    equal_var=False
)

print("\nDescriptive Statistics")
print("New Mean =", new.mean())
print("Loyal Mean =", loyal.mean())

print("\nWelch t-test")
print("T-statistic =", t_h1)
print("P-value =", p_h1)

if p_h1 < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")


# ============================================================
# H2
# UPI vs CREDIT CARD
# Welch Independent t-test
# ============================================================

upi = analysis.loc[
    analysis["payment_type"] == "UPI",
    "order_revenue"
].dropna()

credit_card = analysis.loc[
    analysis["payment_type"] == "Credit Card",
    "order_revenue"
].dropna()

print("\n" + "=" * 70)
print("H2: UPI vs CREDIT CARD ORDER VALUE")
print("=" * 70)

print("\nIndependence assumption:")
print("Each order is treated as an independent observation.")

check_sample_size(upi, "UPI")
check_sample_size(credit_card, "Credit Card")

check_normality(upi, "UPI")
check_normality(credit_card, "Credit Card")

t_h2, p_h2 = stats.ttest_ind(
    upi,
    credit_card,
    equal_var=False
)

print("\nDescriptive Statistics")
print("UPI Mean =", upi.mean())
print("Credit Card Mean =", credit_card.mean())

print("\nWelch t-test")
print("T-statistic =", t_h2)
print("P-value =", p_h2)

if p_h2 < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")


# ============================================================
# H3
# PRODUCT PRICE vs REVENUE
# Chi-square
# ============================================================

order_price = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        avg_unit_price=("unit_price", "mean")
    )
)

h3 = analysis.merge(
    order_price,
    on="order_id",
    how="inner"
)

price_median = h3["avg_unit_price"].median()
revenue_median = h3["order_revenue"].median()

h3["price_category"] = np.where(
    h3["avg_unit_price"] <= price_median,
    "Low Price",
    "High Price"
)

h3["revenue_category"] = np.where(
    h3["order_revenue"] <= revenue_median,
    "Low Revenue",
    "High Revenue"
)

table_h3 = pd.crosstab(
    h3["price_category"],
    h3["revenue_category"]
)

chi_h3, p_h3, dof_h3, expected_h3 = stats.chi2_contingency(
    table_h3
)

print("\n" + "=" * 70)
print("H3: PRODUCT PRICE vs REVENUE")
print("=" * 70)

print("\nIndependence assumption:")
print("Each order is treated as an independent observation.")

print("\nSample size:")
print("N =", len(h3))

print("\nContingency Table:")
print(table_h3)

print("\nExpected Frequencies:")
print(pd.DataFrame(
    expected_h3,
    index=table_h3.index,
    columns=table_h3.columns
))

print("\nMinimum Expected Frequency =", expected_h3.min())

if expected_h3.min() >= 5:
    print("Sample-size assumption: Satisfied.")
else:
    print("Sample-size assumption: NOT satisfied.")

print("\nChi-square =", chi_h3)
print("Degrees of Freedom =", dof_h3)
print("P-value =", p_h3)

if p_h3 < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")


# ============================================================
# H4
# REVIEW SCORE vs REVENUE
# Chi-square
# ============================================================

h4 = analysis.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="inner"
)

h4["review_category"] = pd.cut(
    h4["review_score"],
    bins=[0, 2, 3, 5],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

revenue_median_h4 = h4["order_revenue"].median()

h4["revenue_category"] = np.where(
    h4["order_revenue"] <= revenue_median_h4,
    "Low Revenue",
    "High Revenue"
)

table_h4 = pd.crosstab(
    h4["review_category"],
    h4["revenue_category"]
)

chi_h4, p_h4, dof_h4, expected_h4 = stats.chi2_contingency(
    table_h4
)

print("\n" + "=" * 70)
print("H4: REVIEW SCORE vs REVENUE")
print("=" * 70)

print("\nIndependence assumption:")
print("Each order is treated as an independent observation.")

print("\nSample size:")
print("N =", len(h4))

print("\nContingency Table:")
print(table_h4)

print("\nExpected Frequencies:")
print(pd.DataFrame(
    expected_h4,
    index=table_h4.index,
    columns=table_h4.columns
))

print("\nMinimum Expected Frequency =", expected_h4.min())

if expected_h4.min() >= 5:
    print("Sample-size assumption: Satisfied.")
else:
    print("Sample-size assumption: NOT satisfied.")

print("\nChi-square =", chi_h4)
print("Degrees of Freedom =", dof_h4)
print("P-value =", p_h4)

if p_h4 < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")


# ============================================================
# H5
# PRODUCT CATEGORY vs REVENUE
# Chi-square
# ============================================================

order_product = (
    order_items[
        ["order_id", "product_id"]
    ]
    .drop_duplicates()
)

h5 = analysis.merge(
    order_product,
    on="order_id",
    how="inner"
)

h5 = h5.merge(
    products[
        ["product_id", "Category_name"]
    ],
    on="product_id",
    how="left"
)

h5 = h5.dropna(
    subset=["Category_name", "order_revenue"]
)

revenue_median_h5 = h5["order_revenue"].median()

h5["revenue_category"] = np.where(
    h5["order_revenue"] <= revenue_median_h5,
    "Low Revenue",
    "High Revenue"
)

table_h5 = pd.crosstab(
    h5["Category_name"],
    h5["revenue_category"]
)

chi_h5, p_h5, dof_h5, expected_h5 = stats.chi2_contingency(
    table_h5
)

print("\n" + "=" * 70)
print("H5: PRODUCT CATEGORY vs REVENUE")
print("=" * 70)

print("\nIndependence assumption:")
print("Each order-product combination is treated as an independent observation.")

print("\nSample size:")
print("N =", len(h5))

print("\nNumber of Categories =", h5["Category_name"].nunique())

print("\nMinimum Expected Frequency =", expected_h5.min())

if expected_h5.min() >= 5:
    print("Sample-size assumption: Satisfied.")
else:
    print("Sample-size assumption: Some expected frequencies are below 5.")

print("\nChi-square =", chi_h5)
print("Degrees of Freedom =", dof_h5)
print("P-value =", p_h5)

if p_h5 < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")


# ============================================================
# H6
# LOW DISCOUNT vs HIGH DISCOUNT
# Welch Independent t-test
# ============================================================

order_discount = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        avg_discount=("discount(%)", "mean")
    )
)

h6 = analysis.merge(
    order_discount,
    on="order_id",
    how="inner"
)

discount_median = h6["avg_discount"].median()

h6["discount_group"] = np.where(
    h6["avg_discount"] <= discount_median,
    "Low Discount",
    "High Discount"
)

low_discount = h6.loc[
    h6["discount_group"] == "Low Discount",
    "order_revenue"
].dropna()

high_discount = h6.loc[
    h6["discount_group"] == "High Discount",
    "order_revenue"
].dropna()

print("\n" + "=" * 70)
print("H6: DISCOUNT LEVEL vs REVENUE")
print("=" * 70)

print("\nDiscount Median =", discount_median)

print("\nIndependence assumption:")
print("Each order is treated as an independent observation.")

check_sample_size(low_discount, "Low Discount")
check_sample_size(high_discount, "High Discount")

check_normality(low_discount, "Low Discount")
check_normality(high_discount, "High Discount")

t_h6, p_h6 = stats.ttest_ind(
    low_discount,
    high_discount,
    equal_var=False
)

print("\nDescriptive Statistics")
print("Low Discount Mean =", low_discount.mean())
print("High Discount Mean =", high_discount.mean())

print("\nWelch t-test")
print("T-statistic =", t_h6)
print("P-value =", p_h6)

if p_h6 < 0.05:
    print("Decision: Reject H0")
else:
    print("Decision: Fail to reject H0")


# ============================================================
# FINAL SUMMARY
# ============================================================

results = pd.DataFrame({
    "Hypothesis": [
        "H1: Customer Segment vs Revenue",
        "H2: Payment Type vs Order Value",
        "H3: Product Price vs Revenue",
        "H4: Review Score vs Revenue",
        "H5: Product Category vs Revenue",
        "H6: Discount Level vs Revenue"
    ],

    "Test": [
        "Welch t-test",
        "Welch t-test",
        "Chi-square",
        "Chi-square",
        "Chi-square",
        "Welch t-test"
    ],

    "Statistic": [
        t_h1,
        t_h2,
        chi_h3,
        chi_h4,
        chi_h5,
        t_h6
    ],

    "P_Value": [
        p_h1,
        p_h2,
        p_h3,
        p_h4,
        p_h5,
        p_h6
    ],

    "Decision": [
        "Reject H0" if p_h1 < 0.05 else "Fail to reject H0",
        "Reject H0" if p_h2 < 0.05 else "Fail to reject H0",
        "Reject H0" if p_h3 < 0.05 else "Fail to reject H0",
        "Reject H0" if p_h4 < 0.05 else "Fail to reject H0",
        "Reject H0" if p_h5 < 0.05 else "Fail to reject H0",
        "Reject H0" if p_h6 < 0.05 else "Fail to reject H0"
    ]
})

print("\n\n" + "=" * 80)
print("FINAL STATISTICAL VALIDATION SUMMARY")
print("=" * 80)

print(results.to_string(index=False))

Analysis dataset shape: (66517, 11)

H1: NEW vs LOYAL CUSTOMER REVENUE

Independence assumption:
Each order is treated as an independent observation.

Sample Size Check - New Customers
N = 33455
Result: Sample size is adequate (n >= 30).

Sample Size Check - Loyal Customers
N = 9763
Result: Sample size is adequate (n >= 30).

Normality Check - New Customers
Sample size used: 5000
Shapiro-Wilk statistic: 0.6809869919825802
P-value: 9.831482822379583e-71
Result: Normality assumption is rejected.

Normality Check - Loyal Customers
Sample size used: 5000
Shapiro-Wilk statistic: 0.7036047890929249
P-value: 2.9927103298392814e-69
Result: Normality assumption is rejected.

Descriptive Statistics
New Mean = 14430.442261724706
Loyal Mean = 14222.281847895114

Welch t-test
T-statistic = 1.4265731611638222
P-value = 0.15372176311567592
Decision: Fail to reject H0

H2: UPI vs CREDIT CARD ORDER VALUE

Independence assumption:
Each order is treated as an independent observation.

Sample Size Check -

#### Overall Hypothesis Conclusion:
At a 5% significance level, H1, H2, and H4 failed to reject the null hypothesis, indicating insufficient statistical evidence of significant differences or associations. H3, H5, and H6 rejected the null hypothesis, indicating statistically significant relationships between product price, product category, discount level, and revenue. These results provide statistical evidence to support further business analysis of pricing, product categories, and discount strategies.